In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, cv2, numpy as np
from sklearn.model_selection import train_test_split

frames_dir = "/kaggle/input/frames-masks/frames"
masks_dir  = "/kaggle/input/frames-masks/masks"

def load_data(frames_dir, masks_dir):
    X, Y = [], []
    frames = sorted([f for f in os.listdir(frames_dir) if f.endswith(".png")])
    for img_name in frames:
        img_path  = os.path.join(frames_dir, img_name)
        mask_path = os.path.join(masks_dir, img_name)
        if not os.path.exists(mask_path):
            continue
        img  = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) / 255.0
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0.5).astype(np.float32)
        X.append(img)
        Y.append(mask)
    X = np.expand_dims(np.array(X, dtype=np.float32), -1)
    Y = np.expand_dims(np.array(Y, dtype=np.float32), -1)
    print("✅ Data loaded:", X.shape, Y.shape)
    return X, Y

X, Y = load_data(frames_dir, masks_dir)
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Val:", X_val.shape)


In [ ]:
preds = model.predict(X_val)


In [ ]:
import matplotlib.pyplot as plt
def visualize_predictions(X_val, Y_val, preds, n=5):
    plt.figure(figsize=(12, n * 3))
    for i in range(n):
        idx = np.random.randint(0, len(X_val))
        img = X_val[idx].squeeze()
        gt  = Y_val[idx].squeeze()
        pr  = (preds[idx].squeeze() > 0.5).astype(np.uint8)

        plt.subplot(n, 3, 3*i+1); plt.imshow(img, cmap='gray'); plt.title("Input"); plt.axis('off')
        plt.subplot(n, 3, 3*i+2); plt.imshow(gt, cmap='gray');  plt.title("Ground Truth"); plt.axis('off')
        plt.subplot(n, 3, 3*i+3); plt.imshow(pr, cmap='gray');  plt.title("Predicted"); plt.axis('off')
    plt.tight_layout(); plt.show()

visualize_predictions(X_val, Y_val, preds, 5)


In [ ]:
# ==========================================================
# 1️⃣ Setup and Imports
# ==========================================================
import os, cv2, numpy as np
from tqdm import tqdm
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

print("✅ TensorFlow version:", tf.__version__)

# ==========================================================
# 2️⃣ Define paths
# ==========================================================
frames_dir = "/kaggle/input/frames-masks/frames"
masks_dir  = "/kaggle/input/frames-masks/masks"

# ==========================================================
# 3️⃣ Load frames and masks
# ==========================================================
def load_data(frames_dir, masks_dir):
    X, Y = [], []
    common_files = sorted(list(set(os.listdir(frames_dir)) & set(os.listdir(masks_dir))))
    print(f"Frames: {len(os.listdir(frames_dir))}, Masks: {len(os.listdir(masks_dir))}, Common: {len(common_files)}")

    for img_name in tqdm(common_files, desc="Loading data"):
        img_path = os.path.join(frames_dir, img_name)
        mask_path = os.path.join(masks_dir, img_name)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            continue
        img = cv2.resize(img, (112, 112))
        mask = cv2.resize(mask, (112, 112))
        img = img.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)
        X.append(img)
        Y.append(mask)

    X = np.expand_dims(np.array(X), -1)
    Y = np.expand_dims(np.array(Y), -1)
    return X, Y

X, Y = load_data(frames_dir, masks_dir)
print("✅ Data loaded:", X.shape, Y.shape)

# ==========================================================
# 4️⃣ Train/Validation Split
# ==========================================================
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Val:", X_val.shape)

# ==========================================================
# 5️⃣ Build Deeper U-Net
# ==========================================================
def build_unet(input_shape):
    inputs = layers.Input(input_shape)

    # Encoder
    c1 = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    c1 = layers.Conv2D(32, 3, activation="relu", padding="same")(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation="relu", padding="same")(p1)
    c2 = layers.Conv2D(64, 3, activation="relu", padding="same")(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(p2)
    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(c3)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(p3)
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(c4)

    # Decoder
    u1 = layers.UpSampling2D()(c4)
    u1 = layers.concatenate([u1, c3])
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(u1)
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(c5)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.concatenate([u2, c2])
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(u2)
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(c6)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.concatenate([u3, c1])
    c7 = layers.Conv2D(32, 3, activation="relu", padding="same")(u3)
    c7 = layers.Conv2D(32, 3, activation="relu", padding="same")(c7)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c7)
    return models.Model(inputs, outputs)

# ==========================================================
# 6️⃣ Define Dice Loss
# ==========================================================
def dice_loss(y_true, y_pred, smooth=1):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - ((2. * intersection + smooth) /
                (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth))

# ==========================================================
# 7️⃣ Compile Model
# ==========================================================
model = build_unet((112, 112, 1))
model.compile(optimizer="adam", loss=dice_loss, metrics=["accuracy"])
model.summary()

# ==========================================================
# 8️⃣ Train Model
# ==========================================================
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", save_best_only=True, monitor="val_loss"),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=80,
    batch_size=16,
    callbacks=callbacks
)

# ==========================================================
# 9️⃣ Evaluate Segmentation (IoU)
# ==========================================================
from sklearn.metrics import jaccard_score

preds = model.predict(X_val)
ious = []
for i in range(len(preds)):
    y_true = (Y_val[i].flatten() > 0.5).astype(int)
    y_pred = (preds[i].flatten() > 0.5).astype(int)
    if np.sum(y_true) == 0 and np.sum(y_pred) == 0:
        continue
    ious.append(jaccard_score(y_true, y_pred, zero_division=0))
mean_iou = np.mean(ious) if len(ious) > 0 else 0
print(f"✅ Mean IoU: {mean_iou:.4f}")

# ==========================================================
# 🔟 Calculate EDV, ESV, EF (using first & last frame example)
# ==========================================================
def calculate_volume(mask, pixel_area=1.0):
    return np.sum(mask) * pixel_area

# Example: first and last validation frames
ed_mask = (preds[0] > 0.5).astype(np.uint8)
es_mask = (preds[-1] > 0.5).astype(np.uint8)

EDV = calculate_volume(ed_mask)
ESV = calculate_volume(es_mask)
EF = ((EDV - ESV) / EDV * 100) if EDV > 0 else 0
print(f"💓 EDV: {EDV:.2f}, ESV: {ESV:.2f}, EF: {EF:.2f}%")

# ==========================================================
# 11️⃣ Visualization
# ==========================================================
import matplotlib.pyplot as plt

for i in range(2):
    fig, ax = plt.subplots(1, 3, figsize=(10, 4))
    ax[0].imshow(X_val[i].squeeze(), cmap='gray')
    ax[0].set_title('Input')
    ax[1].imshow(Y_val[i].squeeze(), cmap='gray')
    ax[1].set_title('Ground Truth')
    ax[2].imshow((preds[i] > 0.5).squeeze(), cmap='gray')
    ax[2].set_title('Predicted')
    for a in ax: a.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- 1️⃣ Dice + BCE Loss ---
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

# --- 2️⃣ Data Augmentation ---
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest'
)

train_gen = datagen.flow(X_train, Y_train, batch_size=16)
val_gen = ImageDataGenerator().flow(X_val, Y_val, batch_size=16)

# --- 3️⃣ Build the same U-Net ---
def build_unet(input_shape=(112, 112, 1)):
    inputs = layers.Input(shape=input_shape)

    def conv_block(x, filters):
        x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        return x

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 256)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    c5 = conv_block(p4, 512)

    u6 = layers.UpSampling2D((2, 2))(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = conv_block(u6, 256)

    u7 = layers.UpSampling2D((2, 2))(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = conv_block(u7, 128)

    u8 = layers.UpSampling2D((2, 2))(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = conv_block(u8, 64)

    u9 = layers.UpSampling2D((2, 2))(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = conv_block(u9, 32)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = models.Model(inputs, outputs)
    return model

model = build_unet()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss=bce_dice_loss,
              metrics=['accuracy', dice_coef])

model.summary()

# --- 4️⃣ Callbacks ---
callbacks = [
    ModelCheckpoint('best_unet_model.h5', monitor='val_dice_coef', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_dice_coef', mode='max', patience=10, verbose=1, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

# --- 5️⃣ Train the Model ---
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)

# --- 6️⃣ Evaluate ---
val_preds = model.predict(X_val)
val_preds_bin = (val_preds > 0.5).astype(np.uint8)
iou_score = np.mean([
    np.sum((p & y)) / np.sum((p | y)) if np.sum((p | y)) > 0 else 1
    for p, y in zip(val_preds_bin, Y_val.astype(np.uint8))
])
print(f"✅ Fine-tuned Mean IoU: {iou_score:.4f}")


In [ ]:
# ==========================================================
#  Phase 2️⃣ - EDV, ESV, and EF Calculation
# ==========================================================
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tqdm import tqdm
import matplotlib.pyplot as plt

# ----------------------------
# 1️⃣ Load trained model
# ----------------------------
model_path = "/kaggle/working/best_unet_model.h5"  # update if different
model = load_model(model_path, compile=False)
print("✅ Loaded best model from:", model_path)

# ----------------------------
# 2️⃣ Define your frames folder
# ----------------------------
frames_dir = "/kaggle/input/frames-masks/frames"
frame_files = sorted(os.listdir(frames_dir))

# ----------------------------
# 3️⃣ Helper: predict segmentation mask
# ----------------------------
def predict_mask(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, (112, 112)) / 255.0
    img_resized = np.expand_dims(img_resized, axis=(0, -1))
    pred = model.predict(img_resized)[0, :, :, 0]
    return (pred > 0.5).astype(np.uint8)

# ----------------------------
# 4️⃣ Group frames by video ID
# ----------------------------
from collections import defaultdict
videos = defaultdict(list)
for f in frame_files:
    video_id = f.split("_")[0]
    videos[video_id].append(f)

# ----------------------------
# 5️⃣ Function to compute EF per video
# ----------------------------
def compute_ef_for_video(video_id, frame_list):
    frame_list = sorted(frame_list, key=lambda x: int(x.split("_")[-1].split(".")[0]))
    areas = []
    for f in frame_list:
        mask = predict_mask(os.path.join(frames_dir, f))
        areas.append(np.sum(mask))
    if len(areas) < 2:
        return None

    edv = max(areas)  # largest area → end-diastole
    esv = min(areas)  # smallest area → end-systole
    ef = ((edv - esv) / edv * 100) if edv > 0 else 0
    return edv, esv, ef

# ----------------------------
# 6️⃣ Run on sample subset
# ----------------------------
results = []
for vid, flist in tqdm(list(videos.items())[:5], desc="Computing EF"):
    edv, esv, ef = compute_ef_for_video(vid, flist)
    results.append((vid, edv, esv, ef))
    print(f"{vid} -> EDV:{edv:.1f}, ESV:{esv:.1f}, EF:{ef:.2f}%")

# ----------------------------
# 7️⃣ Optional: visualize one example
# ----------------------------
sample_vid = list(videos.keys())[0]
sample_frames = sorted(videos[sample_vid], key=lambda x: int(x.split("_")[-1].split(".")[0]))

ed_frame = sample_frames[np.argmax([np.sum(predict_mask(os.path.join(frames_dir, f))) for f in sample_frames])]
es_frame = sample_frames[np.argmin([np.sum(predict_mask(os.path.join(frames_dir, f))) for f in sample_frames])]

ed_img = cv2.imread(os.path.join(frames_dir, ed_frame), cv2.IMREAD_GRAYSCALE)
es_img = cv2.imread(os.path.join(frames_dir, es_frame), cv2.IMREAD_GRAYSCALE)
ed_mask = predict_mask(os.path.join(frames_dir, ed_frame))
es_mask = predict_mask(os.path.join(frames_dir, es_frame))

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(ed_img, cmap='gray')
plt.imshow(ed_mask, cmap='Reds', alpha=0.4)
plt.title(f"ED Frame: {ed_frame}")

plt.subplot(1,2,2)
plt.imshow(es_img, cmap='gray')
plt.imshow(es_mask, cmap='Blues', alpha=0.4)
plt.title(f"ES Frame: {es_frame}")
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tqdm import tqdm
import pandas as pd

# --- Paths ---
frames_dir = "/kaggle/input/frames-masks/frames"
masks_dir = "/kaggle/input/frames-masks/masks"
model_path = "/kaggle/working/best_unet_model.h5"
output_csv = "/kaggle/working/ef_results_all.csv"

# --- Load model ---
model = tf.keras.models.load_model(model_path, compile=False)
print("✅ Loaded model for EF computation")

# --- Helper functions ---
def preprocess_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (112, 112))
    img = img / 255.0
    img = np.expand_dims(img, axis=(0, -1))
    return img

def compute_volume(mask):
    # mask values: 0 or 1, each pixel = area unit
    return np.sum(mask)

# --- Get unique video IDs ---
video_ids = sorted(list(set([f.split("_")[0] for f in os.listdir(frames_dir) if f.endswith(".png")])))

results = []

for vid in tqdm(video_ids, desc="Computing EF"):
    # get all frame paths for this video
    frame_files = sorted([f for f in os.listdir(frames_dir) if f.startswith(vid)])
    edv, esv = 0, 0
    ed_frame, es_frame = None, None
    max_vol, min_vol = -np.inf, np.inf

    for frame_file in frame_files:
        frame_path = os.path.join(frames_dir, frame_file)
        frame = preprocess_image(frame_path)

        # Predict mask
        pred = model.predict(frame, verbose=0)[0, :, :, 0]
        pred_bin = (pred > 0.5).astype(np.uint8)

        volume = compute_volume(pred_bin)

        if volume > max_vol:
            max_vol = volume
            ed_frame = frame_file
        if volume < min_vol:
            min_vol = volume
            es_frame = frame_file

    edv = max_vol
    esv = min_vol
    ef = ((edv - esv) / edv * 100) if edv > 0 else 0
    results.append([vid, round(edv, 2), round(esv, 2), round(ef, 2)])

# --- Save all results ---
df = pd.DataFrame(results, columns=["Video_ID", "EDV", "ESV", "EF"])
df.to_csv(output_csv, index=False)
print(f"📁 Saved all EF results to: {output_csv}")
print(df.head())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score

# ==========================================================
# Paths
# ==========================================================
predicted_csv = "/kaggle/working/ef_results_all.csv"
filelist_csv = "/kaggle/input/heartdatabase/EchoNet-Dynamic/FileList.csv"

# ==========================================================
# Load data
# ==========================================================
pred_df = pd.read_csv(predicted_csv)
gt_df = pd.read_csv(filelist_csv)

print(f"✅ Loaded predicted EF results: {len(pred_df)} rows")
print(f"✅ Loaded ground truth: {len(gt_df)} rows")

# ==========================================================
# Merge safely and rename columns
# ==========================================================
merged = pred_df.merge(gt_df, left_on="Video_ID", right_on="FileName", how="inner")
print(f"✅ Merged dataset: {len(merged)} matched videos")

# Rename overlapping columns
merged.rename(columns={
    "EF_x": "Pred_EF",
    "EF_y": "True_EF",
    "EDV_y": "True_EDV",
    "ESV_y": "True_ESV",
    "EDV_x": "Pred_EDV",
    "ESV_x": "Pred_ESV"
}, inplace=True)

# ==========================================================
# Compute metrics
# ==========================================================
mae = mean_absolute_error(merged["True_EF"], merged["Pred_EF"])
r2 = r2_score(merged["True_EF"], merged["Pred_EF"])

print(f"📊 Mean Absolute Error (EF): {mae:.2f}")
print(f"📈 R² Score: {r2:.3f}")

# ==========================================================
# Visualization 1: Predicted vs True EF
# ==========================================================
plt.figure(figsize=(6,6))
sns.regplot(data=merged, x="True_EF", y="Pred_EF", scatter_kws={"alpha":0.7})
plt.plot([0, 100], [0, 100], "r--", label="Ideal")
plt.xlabel("True EF (%)")
plt.ylabel("Predicted EF (%)")
plt.title("Predicted vs True Ejection Fraction")
plt.legend()
plt.grid(True)
plt.show()

# ==========================================================
# Visualization 2: EF Distribution Comparison
# ==========================================================
plt.figure(figsize=(8,5))
sns.histplot(merged["True_EF"], color="green", label="True EF", kde=True, stat="density", alpha=0.5)
sns.histplot(merged["Pred_EF"], color="blue", label="Predicted EF", kde=True, stat="density", alpha=0.5)
plt.xlabel("Ejection Fraction (%)")
plt.title("EF Distribution: True vs Predicted")
plt.legend()
plt.grid(True)
plt.show()

# ==========================================================
# Visualization 3: Bland–Altman Plot
# ==========================================================
mean_ef = (merged["True_EF"] + merged["Pred_EF"]) / 2
diff_ef = merged["Pred_EF"] - merged["True_EF"]

plt.figure(figsize=(7,5))
plt.scatter(mean_ef, diff_ef, alpha=0.6)
plt.axhline(diff_ef.mean(), color="red", linestyle="--", label="Mean bias")
plt.axhline(diff_ef.mean() + 1.96*diff_ef.std(), color="gray", linestyle="--", label="±1.96 SD")
plt.axhline(diff_ef.mean() - 1.96*diff_ef.std(), color="gray", linestyle="--")
plt.xlabel("Mean EF (%)")
plt.ylabel("Predicted - True EF (%)")
plt.title("Bland–Altman Plot for EF Agreement")
plt.legend()
plt.grid(True)
plt.show()

# ==========================================================
# Save comparison
# ==========================================================
merged.to_csv("/kaggle/working/ef_comparison.csv", index=False)
print("💾 Saved EF comparison to: /kaggle/working/ef_comparison.csv")


In [ ]:
import matplotlib.pyplot as plt
import random
import cv2
import os
import numpy as np
from tqdm import tqdm

# Paths
frames_dir = "/kaggle/input/frames-masks/frames"
masks_dir = "/kaggle/input/frames-masks/masks"
model_path = "/kaggle/working/best_unet_model.h5"

# Load best model
from tensorflow.keras.models import load_model
model = load_model(model_path, compile=False)
print(f"✅ Loaded model from {model_path}")

# Pick some random test frames
all_images = sorted(os.listdir(frames_dir))
sample_images = random.sample(all_images, 5)  # show 5 samples

for img_name in sample_images:
    frame_path = os.path.join(frames_dir, img_name)
    mask_path = os.path.join(masks_dir, img_name)

    # Load frame and ground-truth mask
    frame = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    frame_input = frame.astype(np.float32) / 255.0
    frame_input = np.expand_dims(frame_input, axis=(0, -1))

    # Predict mask
    pred_mask = model.predict(frame_input)[0, :, :, 0]
    pred_mask = (pred_mask > 0.5).astype(np.uint8) * 255

    # Plot side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(f"🔍 {img_name}", fontsize=12)

    axes[0].imshow(frame, cmap='gray')
    axes[0].set_title("Original Frame")

    axes[1].imshow(gt_mask, cmap='gray')
    axes[1].set_title("Ground Truth Mask")

    axes[2].imshow(frame, cmap='gray')
    axes[2].imshow(pred_mask, alpha=0.5, cmap='jet')
    axes[2].set_title("Predicted Mask Overlay")

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
